# SQL for Beginners — Comprehensive Hands-On Notebook

Welcome! This notebook teaches SQL from the ground up using a small, realistic dataset.

You will learn how to:

- understand what databases and tables are
- query data with `SELECT`
- filter rows with `WHERE`
- sort and limit results
- summarize data with aggregate functions
- group data with `GROUP BY` and filter groups with `HAVING`
- combine tables with `JOIN`
- write subqueries and common table expressions (CTEs)
- use `CASE`, handle `NULL`, and work with text and dates
- modify data with `INSERT`, `UPDATE`, and `DELETE`
- create and modify database structures with DDL commands
- understand constraints, views, transactions, indexes, and window functions

This notebook uses **SQLite** through Python, so you can run everything locally without installing a database server.

## How to use this notebook

1. Run the setup cells in order.
2. Read the explanation in each section.
3. Run the SQL examples.
4. Try the practice exercises before revealing the solutions.

> SQL syntax varies a little across database systems like PostgreSQL, MySQL, SQL Server, and Oracle.  
> The core ideas are the same, and this notebook focuses on the fundamentals that transfer well.

In [ ]:

import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

def run_sql(query, params=None):
    """Run a SQL query and return a DataFrame when possible."""
    if params is None:
        params = ()
    query_stripped = query.strip().lower()
    cur = conn.cursor()
    cur.execute(query, params)

    if query_stripped.startswith(("select", "with", "pragma", "explain")):
        rows = cur.fetchall()
        cols = [desc[0] for desc in cur.description] if cur.description else []
        return pd.DataFrame(rows, columns=cols)
    else:
        conn.commit()
        print("Query executed successfully.")

## 1. What is SQL?

**SQL** stands for **Structured Query Language**. It is used to work with data in **relational databases**.

A relational database organizes data into **tables**:
- each **row** is one record
- each **column** is one attribute about that record

Example:
- `customers` table
- `orders` table
- `products` table

SQL helps you:
- read data
- filter data
- summarize data
- combine data from multiple tables
- add, update, or delete records

## 2. Build a small sample database

We will create four tables:

- `customers`
- `products`
- `orders`
- `order_items`

This is a common ecommerce-style schema and is great for practicing SQL.

In [ ]:

schema_sql = '''
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    city TEXT,
    state TEXT,
    signup_date TEXT,
    email TEXT UNIQUE
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
'''
conn.executescript(schema_sql)
print("Tables created.")

In [ ]:

seed_sql = '''
INSERT INTO customers (customer_id, first_name, last_name, city, state, signup_date, email) VALUES
(1, 'Ava', 'Johnson', 'Baltimore', 'MD', '2025-01-10', 'ava@example.com'),
(2, 'Liam', 'Smith', 'Towson', 'MD', '2025-02-14', 'liam@example.com'),
(3, 'Noah', 'Brown', 'Columbia', 'MD', '2025-02-20', 'noah@example.com'),
(4, 'Emma', 'Davis', 'Arlington', 'VA', '2025-03-05', 'emma@example.com'),
(5, 'Olivia', 'Wilson', 'Silver Spring', 'MD', '2025-03-18', 'olivia@example.com'),
(6, 'Sophia', 'Miller', 'Alexandria', 'VA', '2025-04-01', 'sophia@example.com');

INSERT INTO products (product_id, product_name, category, price, stock_quantity) VALUES
(1, 'Laptop', 'Electronics', 1200.00, 15),
(2, 'Mouse', 'Electronics', 25.00, 100),
(3, 'Keyboard', 'Electronics', 45.00, 60),
(4, 'Desk Chair', 'Furniture', 180.00, 20),
(5, 'Notebook', 'Office Supplies', 5.00, 200),
(6, 'Pen Set', 'Office Supplies', 12.00, 150),
(7, 'Monitor', 'Electronics', 250.00, 30),
(8, 'Standing Desk', 'Furniture', 400.00, 10);

INSERT INTO orders (order_id, customer_id, order_date, status) VALUES
(1, 1, '2025-04-10', 'Shipped'),
(2, 2, '2025-04-12', 'Pending'),
(3, 1, '2025-04-15', 'Shipped'),
(4, 3, '2025-04-18', 'Cancelled'),
(5, 4, '2025-04-19', 'Shipped'),
(6, 5, '2025-04-22', 'Pending'),
(7, 2, '2025-04-25', 'Shipped'),
(8, 6, '2025-04-27', 'Shipped');

INSERT INTO order_items (order_item_id, order_id, product_id, quantity, unit_price) VALUES
(1, 1, 1, 1, 1200.00),
(2, 1, 2, 2, 25.00),
(3, 2, 5, 10, 5.00),
(4, 2, 6, 3, 12.00),
(5, 3, 7, 2, 250.00),
(6, 3, 3, 1, 45.00),
(7, 4, 4, 1, 180.00),
(8, 5, 8, 1, 400.00),
(9, 5, 2, 1, 25.00),
(10, 6, 5, 20, 5.00),
(11, 7, 1, 1, 1150.00),
(12, 7, 7, 1, 250.00),
(13, 8, 4, 1, 180.00),
(14, 8, 5, 5, 5.00);
'''
conn.executescript(seed_sql)
print("Sample data inserted.")

In [ ]:

run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

## 4. View a table with SELECT

The most basic SQL query reads data from a table.

### Syntax

```sql
SELECT column1, column2
FROM table_name;
```

To return every column, you can use `*`:

```sql
SELECT *
FROM table_name;
```

In [ ]:
## 5. Rename columns with aliases

Aliases make results easier to read.

### Syntax

```sql
SELECT column_name AS new_name
FROM table_name;
```

## 6. Remove duplicates with `DISTINCT`

Use `DISTINCT` when you want unique values.

In [ ]:

run_sql("""
SELECT first_name, last_name, city
FROM customers;
""")

## 7. Filter rows with `WHERE`

`WHERE` filters rows before they are returned.

### Common comparison operators

- `=` equal to
- `!=` or `<>` not equal to
- `>` greater than
- `<` less than
- `>=` greater than or equal to
- `<=` less than or equal to

In [ ]:

run_sql("""
SELECT
    first_name AS first,
    last_name AS last,
    signup_date AS joined_on
FROM customers;
""")

## 8. Sort results with `ORDER BY`

Sort ascending with `ASC` and descending with `DESC`.

In [ ]:

run_sql("""
SELECT DISTINCT state
FROM customers;
""")

In [ ]:

run_sql("""
SELECT DISTINCT category
FROM products;
""")

## 9. Limit the number of rows with `LIMIT`

Useful for previews and top-N style queries.

In [ ]:

run_sql("""
SELECT *
FROM products
WHERE price > 100;
""")

### Multiple conditions with `AND` and `OR`

In [ ]:

run_sql("""
SELECT *
FROM customers
WHERE state = 'MD' AND city = 'Baltimore';
""")

In [ ]:

run_sql("""
SELECT *
FROM orders
WHERE status = 'Pending' OR status = 'Cancelled';
""")

### Filter a range with `BETWEEN`

In [ ]:

run_sql("""
SELECT product_name, price
FROM products
WHERE price BETWEEN 20 AND 300;
""")

### Match a list with `IN`

In [ ]:

run_sql("""
SELECT first_name, last_name, city
FROM customers
WHERE city IN ('Baltimore', 'Towson', 'Columbia');
""")

### Search text with `LIKE`

Pattern wildcards:
- `%` = any number of characters
- `_` = exactly one character

In [ ]:

run_sql("""
SELECT *
FROM products
WHERE product_name LIKE '%desk%';
""")

### Check for missing values with `IS NULL`

Never use `= NULL`. Use:
- `IS NULL`
- `IS NOT NULL`

In [ ]:

run_sql("""
SELECT *
FROM customers
WHERE city IS NOT NULL;
""")

## 7. Sort results with `ORDER BY`

Sort ascending with `ASC` and descending with `DESC`.

In [ ]:

run_sql("""
SELECT product_name, price
FROM products
ORDER BY price ASC;
""")

In [ ]:

run_sql("""
SELECT product_name, price
FROM products
ORDER BY price DESC;
""")

You can sort by multiple columns too.

In [ ]:

run_sql("""
SELECT state, city, first_name, last_name
FROM customers
ORDER BY state ASC, city ASC, last_name ASC;
""")

## 8. Limit the number of rows with `LIMIT`

Useful for previews and top-N style queries.

In [ ]:

run_sql("""
SELECT *
FROM products
ORDER BY price DESC
LIMIT 3;
""")

## 9. Create calculated columns

You can compute values directly in SQL.

In [ ]:

run_sql("""
SELECT
    product_name,
    price,
    price * 0.10 AS tax_estimate,
    price * 1.10 AS price_with_tax
FROM products;
""")

## 10. Aggregate functions

Aggregate functions summarize many rows into one value.

Common ones:
- `COUNT()`
- `SUM()`
- `AVG()`
- `MIN()`
- `MAX()`

In [ ]:

run_sql("""
SELECT
    COUNT(*) AS total_products,
    AVG(price) AS avg_price,
    MIN(price) AS cheapest_product,
    MAX(price) AS most_expensive_product
FROM products;
""")

In [ ]:

run_sql("""
SELECT
    SUM(quantity * unit_price) AS gross_sales
FROM order_items;
""")

## 11. Group data with `GROUP BY`

`GROUP BY` splits rows into groups, then aggregates each group.

In [ ]:

run_sql("""
SELECT
    category,
    COUNT(*) AS product_count,
    AVG(price) AS avg_price
FROM products
GROUP BY category;
""")

In [ ]:

run_sql("""
SELECT
    status,
    COUNT(*) AS num_orders
FROM orders
GROUP BY status;
""")

### Important rule

If you use `GROUP BY`, then every selected column must be:
- included in the `GROUP BY`, or
- wrapped in an aggregate function

## 12. Filter grouped results with `HAVING`

`WHERE` filters rows **before grouping**.  
`HAVING` filters groups **after grouping**.

In [ ]:

run_sql("""
SELECT
    category,
    COUNT(*) AS product_count,
    AVG(price) AS avg_price
FROM products
GROUP BY category
HAVING AVG(price) > 100;
""")

## 13. SQL execution order (conceptually)

This is a useful mental model:

1. `FROM`
2. `JOIN`
3. `WHERE`
4. `GROUP BY`
5. `HAVING`
6. `SELECT`
7. `ORDER BY`
8. `LIMIT`

This helps explain why you cannot usually use a `SELECT` alias inside `WHERE`.

## 14. Combine tables with `JOIN`

Databases often store related data in separate tables.  
`JOIN` combines them.

### Main join types
- `INNER JOIN`
- `LEFT JOIN`
- `RIGHT JOIN` (not supported in SQLite)
- `FULL OUTER JOIN` (not supported directly in SQLite)

### `INNER JOIN`

Returns only rows that match in both tables.

In [ ]:

run_sql("""
SELECT
    o.order_id,
    o.order_date,
    o.status,
    c.first_name,
    c.last_name
FROM orders o
INNER JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")

### Join three or more tables

This is very common in real-world SQL.

In [ ]:

run_sql("""
SELECT
    o.order_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    p.product_name,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN products p
    ON oi.product_id = p.product_id
ORDER BY o.order_id, p.product_name;
""")

### `LEFT JOIN`

Returns all rows from the left table, even when there is no match on the right.

In [ ]:

# Add a customer with no orders so LEFT JOIN is easier to see
run_sql("""
INSERT INTO customers (customer_id, first_name, last_name, city, state, signup_date, email)
VALUES (7, 'Mia', 'Taylor', 'Rockville', 'MD', '2025-05-01', 'mia@example.com');
""")

In [ ]:

run_sql("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    o.order_id,
    o.order_date
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id;
""")

Notice that customers without orders still appear, with `NULL` values from the `orders` table.

## 15. Summarize revenue by customer

Let's combine joins, grouping, and aggregation.

In [ ]:

run_sql("""
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, customer_name
ORDER BY total_spent DESC;
""")

## 16. Subqueries

A subquery is a query inside another query.

### Example: products priced above the average price

In [ ]:

run_sql("""
SELECT
    product_name,
    price
FROM products
WHERE price > (
    SELECT AVG(price)
    FROM products
)
ORDER BY price DESC;
""")

### Example: customers who have placed at least one order

In [ ]:

run_sql("""
SELECT *
FROM customers
WHERE customer_id IN (
    SELECT DISTINCT customer_id
    FROM orders
);
""")

## 17. Common Table Expressions (CTEs)

A CTE uses `WITH` to create a temporary named result set.

CTEs make longer queries easier to read.

In [ ]:

run_sql("""
WITH customer_totals AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS total_spent
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.customer_id
)
SELECT
    c.first_name,
    c.last_name,
    ROUND(ct.total_spent, 2) AS total_spent
FROM customer_totals ct
JOIN customers c
    ON ct.customer_id = c.customer_id
ORDER BY total_spent DESC;
""")

## 18. Use `CASE` for conditional logic

`CASE` works like an if/else expression inside SQL.

In [ ]:

run_sql("""
SELECT
    product_name,
    price,
    CASE
        WHEN price >= 500 THEN 'Premium'
        WHEN price >= 100 THEN 'Mid-range'
        ELSE 'Budget'
    END AS price_band
FROM products
ORDER BY price DESC;
""")

## 19. Handle missing values with `COALESCE`

`COALESCE(value1, value2, ...)` returns the first non-NULL value.

In [ ]:

run_sql("""
SELECT
    customer_id,
    first_name,
    COALESCE(city, 'Unknown') AS city
FROM customers;
""")

## 20. String functions

Common text operations include:
- concatenate text
- uppercase/lowercase
- substring
- trim spaces
- length

In [ ]:

run_sql("""
SELECT
    first_name,
    last_name,
    first_name || ' ' || last_name AS full_name,
    UPPER(state) AS state_upper,
    LENGTH(email) AS email_length
FROM customers;
""")

## 21. Date handling in SQLite

SQLite stores dates as text in formats like `YYYY-MM-DD`.  
It also includes date functions.

In [ ]:

run_sql("""
SELECT
    order_id,
    order_date,
    DATE(order_date, '+7 day') AS follow_up_date
FROM orders;
""")

In [ ]:

run_sql("""
SELECT
    STRFTIME('%Y-%m', order_date) AS order_month,
    COUNT(*) AS num_orders
FROM orders
GROUP BY STRFTIME('%Y-%m', order_date)
ORDER BY order_month;
""")

## 22. Data modification: `INSERT`, `UPDATE`, `DELETE`

These commands change data.

### `INSERT`
Add a new row.

In [ ]:

run_sql("""
INSERT INTO products (product_id, product_name, category, price, stock_quantity)
VALUES (9, 'Webcam', 'Electronics', 85.00, 40);
""")

In [ ]:

run_sql("SELECT * FROM products WHERE product_id = 9;")

### `UPDATE`
Change existing rows.

In [ ]:

run_sql("""
UPDATE products
SET stock_quantity = stock_quantity - 5
WHERE product_id = 9;
""")

In [ ]:

run_sql("SELECT * FROM products WHERE product_id = 9;")

### `DELETE`
Remove rows.

Be careful: forgetting a `WHERE` clause can delete every row in a table.

In [ ]:

run_sql("""
DELETE FROM products
WHERE product_id = 9;
""")

In [ ]:

run_sql("SELECT * FROM products WHERE product_id = 9;")

## 23. Table constraints

Constraints help protect data quality.

Common constraints:
- `PRIMARY KEY`
- `NOT NULL`
- `UNIQUE`
- `FOREIGN KEY`
- `CHECK`
- `DEFAULT`

In [ ]:

run_sql("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    department TEXT NOT NULL,
    salary REAL CHECK (salary >= 0),
    active INTEGER DEFAULT 1
);
""")

In [ ]:

run_sql("""
PRAGMA table_info(employees);
""")

## 24. Views

A **view** is a saved query you can treat like a table.

In [ ]:

run_sql("""
CREATE VIEW IF NOT EXISTS order_summary AS
SELECT
    o.order_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    o.order_date,
    o.status,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS order_total
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.order_id, customer_name, o.order_date, o.status;
""")

In [ ]:

run_sql("SELECT * FROM order_summary ORDER BY order_total DESC;")

## 25. Transactions

A transaction groups statements so they succeed or fail together.

This matters when several changes must stay consistent.

In [ ]:

# Example transaction
try:
    conn.execute("BEGIN")
    conn.execute("UPDATE products SET stock_quantity = stock_quantity - 1 WHERE product_id = 1")
    conn.execute("UPDATE products SET stock_quantity = stock_quantity + 1 WHERE product_id = 2")
    conn.commit()
    print("Transaction committed.")
except Exception as e:
    conn.rollback()
    print("Transaction rolled back:", e)

## 26. Indexes

An **index** can make queries faster, especially on large tables.

Tradeoff:
- reads can be faster
- writes can be slightly slower
- indexes use storage

In [ ]:

run_sql("CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders(customer_id);")

In [ ]:

run_sql("""
EXPLAIN QUERY PLAN
SELECT *
FROM orders
WHERE customer_id = 2;
""")

You do not need to memorize the execution plan output yet.  
For now, just know that indexes help databases find rows faster.

## 27. Window functions

Window functions perform calculations across related rows without collapsing them into one row per group.

They are extremely useful once you move beyond the basics.

In [ ]:

run_sql("""
SELECT
    o.order_id,
    o.customer_id,
    SUM(oi.quantity * oi.unit_price) AS order_total,
    RANK() OVER (
        PARTITION BY o.customer_id
        ORDER BY SUM(oi.quantity * oi.unit_price) DESC
    ) AS order_rank_for_customer
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY o.order_id, o.customer_id
ORDER BY o.customer_id, order_rank_for_customer;
""")

## 28. Putting it all together

Here is a more realistic analysis query:

**Question:** What are the top customers by revenue, and how many orders have they placed?

In [ ]:

run_sql("""
WITH order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id
),
customer_summary AS (
    SELECT
        customer_id,
        COUNT(*) AS num_orders,
        SUM(order_total) AS revenue
    FROM order_totals
    GROUP BY customer_id
)
SELECT
    c.first_name || ' ' || c.last_name AS customer_name,
    cs.num_orders,
    ROUND(cs.revenue, 2) AS revenue
FROM customer_summary cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY revenue DESC;
""")

## 29. Practice exercises

Try these on your own before looking at the solutions below.

1. Show all products in the `Electronics` category.
2. Show customers from Maryland (`MD`) ordered by last name.
3. Count how many customers are in each state.
4. Show each order with the customer's full name.
5. Find the total revenue for each order.
6. Find the average product price by category.
7. Show only categories whose average price is greater than 100.
8. Find customers who have never placed an order.
9. Show the most expensive product in each category.
10. Rank each customer's orders from highest total to lowest total.

## 3. DDL Commands

DDL stands for Data Definition Language. It is used to create, modify, and delete database objects like tables, indexes, and views.

### CREATE TABLE

The `CREATE TABLE` statement defines a new table with columns and constraints.

In [ ]:
run_sql(\"\"\"\nCREATE TABLE temp_departments (\n    dept_id INTEGER PRIMARY KEY,\n    dept_name TEXT NOT NULL,\n    location TEXT\n);\n\"\"\")

### ALTER TABLE

The `ALTER TABLE` statement modifies an existing table, such as adding or dropping columns.

In [ ]:
run_sql(\"\"\"\nALTER TABLE temp_departments ADD COLUMN budget REAL DEFAULT 0;\n\"\"\")\n\nrun_sql(\"PRAGMA table_info(temp_departments);\")

### DROP TABLE

The `DROP TABLE` statement deletes a table and all its data.

In [ ]:
run_sql(\"DROP TABLE temp_departments;\")

## 23. Data modification: `INSERT`, `UPDATE`, `DELETE`

These commands change data.

### `INSERT`
Add a new row.

In [ ]:

# 1. Show all products in the Electronics category.
run_sql("""
SELECT *
FROM products
WHERE category = 'Electronics';
""")

In [ ]:

# 2. Show customers from Maryland (MD) ordered by last name.
run_sql("""
SELECT *
FROM customers
WHERE state = 'MD'
ORDER BY last_name;
""")

In [ ]:

# 3. Count how many customers are in each state.
run_sql("""
SELECT
    state,
    COUNT(*) AS customer_count
FROM customers
GROUP BY state;
""")

In [ ]:

# 4. Show each order with the customer's full name.
run_sql("""
SELECT
    o.order_id,
    o.order_date,
    c.first_name || ' ' || c.last_name AS customer_name
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
ORDER BY o.order_id;
""")

In [ ]:

# 5. Find the total revenue for each order.
run_sql("""
SELECT
    order_id,
    ROUND(SUM(quantity * unit_price), 2) AS order_total
FROM order_items
GROUP BY order_id
ORDER BY order_total DESC;
""")

In [ ]:

# 6. Find the average product price by category.
run_sql("""
SELECT
    category,
    AVG(price) AS avg_price
FROM products
GROUP BY category;
""")

In [ ]:

# 7. Show only categories whose average price is greater than 100.
run_sql("""
SELECT
    category,
    AVG(price) AS avg_price
FROM products
GROUP BY category
HAVING AVG(price) > 100;
""")

In [ ]:

# 8. Find customers who have never placed an order.
run_sql("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
""")

In [ ]:

# 9. Show the most expensive product in each category.
run_sql("""
SELECT
    p1.category,
    p1.product_name,
    p1.price
FROM products p1
WHERE p1.price = (
    SELECT MAX(p2.price)
    FROM products p2
    WHERE p2.category = p1.category
)
ORDER BY p1.category;
""")

In [ ]:

# 10. Rank each customer's orders from highest total to lowest total.
run_sql("""
WITH order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id
)
SELECT
    customer_id,
    order_id,
    order_total,
    RANK() OVER (
        PARTITION BY customer_id
        ORDER BY order_total DESC
    ) AS order_rank
FROM order_totals
ORDER BY customer_id, order_rank;
""")

## 24. Table constraints

Constraints help protect data quality.

Common constraints:
- `PRIMARY KEY`
- `NOT NULL`
- `UNIQUE`
- `FOREIGN KEY`
- `CHECK`
- `DEFAULT`

## 25. Views

A **view** is a saved query you can treat like a table.

## 26. Transactions

A transaction groups statements so they succeed or fail together.

This matters when several changes must stay consistent.

## 35. Final recap

You learned how to:
- query data
- filter and sort it
- aggregate and group it
- join multiple tables
- use subqueries, CTEs, and conditional logic
- modify data
- create and modify database structures with DDL commands
- understand constraints, views, transactions, indexes, and window functions

That is a strong beginner foundation.

Keep practicing by writing your own questions against the sample database:
- Which customers spent more than 500?
- Which category has the highest revenue?
- Which order had the most items?
- Which city has the most customers?

In [ ]:

# Optional: a blank cell for your own experiments.
# Try writing your own SQL below.

query = '''
SELECT 'Start practicing here!' AS message;
'''
run_sql(query)